# Embeddings lokaal maken en naar Snowflake schrijven

Alternatief voor `02_embeddings_snowflake.ipynb`, zolang `AI_EMBED` niet beschikbaar is op het trial-account.
Hetzelfde model (`snowflake-arctic-embed-l-v2.0`, open weights via Hugging Face) draait hier lokaal; het resultaat komt in dezelfde tabel `FRUTA.PUBLIC.RECORD_EMBEDDINGS` (`NOTE_ID`, `EMBEDDING`).

Bij opnieuw draaien worden alleen embeddings gemaakt voor rapportages die er nog geen hebben.

In [ ]:
import json
import os

from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from snowflake.snowpark import Session

EMBEDDING_MODEL = "Snowflake/snowflake-arctic-embed-l-v2.0"
EMBEDDING_DIMENSION = 1024

load_dotenv()
session = Session.builder.configs(
    {
        "account": os.getenv("SNOWFLAKE_ACCOUNT"),
        "user": os.getenv("SNOWFLAKE_USER"),
        "password": os.getenv("SNOWFLAKE_PASSWORD"),
        "warehouse": "COMPUTE_WH",
        "database": "FRUTA",
        "schema": "PUBLIC",
    }
).create()

In [ ]:
# Haal de rapportages op die nog geen embedding hebben
table_exists = len(session.sql("SHOW TABLES LIKE 'RECORD_EMBEDDINGS'").collect()) > 0

sql = "SELECT NOTE_ID, NOTE FROM RECORDS r WHERE NOTE IS NOT NULL AND NOTE != ''"
if table_exists:
    sql += " AND NOT EXISTS (SELECT 1 FROM RECORD_EMBEDDINGS e WHERE e.NOTE_ID = r.NOTE_ID)"

df = session.sql(sql).to_pandas()
print(f"{len(df)} rapportages zonder embedding opgehaald.")

In [ ]:
# Laad het model (eerste keer: model wordt gedownload, ca. 2 GB)
model = SentenceTransformer(EMBEDDING_MODEL)

In [ ]:
# Maak de embeddings en schrijf naar Snowflake:
# eerst als JSON-tekst in een tijdelijke tabel, dan omzetten naar VECTOR
if df.empty:
    print("Alle rapportages hebben al een embedding.")
else:
    embeddings = model.encode(df["NOTE"].tolist(), batch_size=32, show_progress_bar=True)
    df["EMBEDDING"] = [json.dumps(e.tolist()) for e in embeddings]
    session.write_pandas(
        df[["NOTE_ID", "EMBEDDING"]],
        "RECORD_EMBEDDINGS_STAGING",
        auto_create_table=True,
        overwrite=True,
        table_type="temporary",
    )

    select = f"""
        SELECT
            NOTE_ID,
            PARSE_JSON(EMBEDDING)::ARRAY::VECTOR(FLOAT, {EMBEDDING_DIMENSION}) AS EMBEDDING
        FROM RECORD_EMBEDDINGS_STAGING
    """
    if table_exists:
        session.sql(f"INSERT INTO RECORD_EMBEDDINGS (NOTE_ID, EMBEDDING) {select}").collect()
    else:
        session.sql(f"CREATE TABLE RECORD_EMBEDDINGS AS {select}").collect()
    print(f"{len(df)} embeddings toegevoegd.")

In [ ]:
# Controle: aantal records en aantal embeddings
session.sql(
    """
    SELECT
        (SELECT COUNT(*) FROM RECORDS) AS RECORDS,
        (SELECT COUNT(*) FROM RECORD_EMBEDDINGS) AS EMBEDDINGS
    """
).show()

In [ ]:
# Test: zoek de rapportages die het meest lijken op een vraag.
# Arctic verwacht bij zoekvragen de prompt 'query'.
query = "Mevrouw is gevallen"
query_embedding = model.encode(query, prompt_name="query")

session.sql(
    f"""
    SELECT
        r.NOTE_ID,
        r.NOTE,
        VECTOR_COSINE_SIMILARITY(
            e.EMBEDDING,
            PARSE_JSON(?)::ARRAY::VECTOR(FLOAT, {EMBEDDING_DIMENSION})
        ) AS SCORE
    FROM RECORD_EMBEDDINGS e
    JOIN RECORDS r ON r.NOTE_ID = e.NOTE_ID
    ORDER BY SCORE DESC
    LIMIT 10
    """,
    params=[json.dumps(query_embedding.tolist())],
).to_pandas()

In [ ]:
session.close()